In [ ]:
# MDS of all ordered pairs (b3, from CSV) at all requested steps, with Kruskal stress-1 and (raw) Kruskal stress
import numpy as np
import matplotlib.pyplot as plt
from sklearn import manifold
from sklearn.metrics import euclidean_distances
from scipy.stats import pearsonr
import pandas as pd

# Use CSV-loaded b3 data if available; else load from gz in this folder
if 'all_b3s' in globals():
    b3_avg = all_b3s  # shape: (time_steps, items_n, b3_size) already averaged across seeds
else:
    def load_b3_mean_from_csv_gz(csv_gz_path):
        df = pd.read_csv(csv_gz_path, compression='gzip')
        seed_cols = [c for c in df.columns if c.startswith('seed_')]
        # Mean across seeds per (time_step, item, unit)
        grouped = df.groupby(['time_step', 'item', 'unit'])[seed_cols].mean()
        mean_series = grouped.mean(axis=1)  # average across seed columns
        time_steps = int(df['time_step'].max()) + 1
        items_n_inferred = int(df['item'].max()) + 1
        units = int(df['unit'].max()) + 1
        arr = np.zeros((time_steps, items_n_inferred, units))
        for (t, i, u), val in mean_series.items():
            arr[int(t), int(i), int(u)] = float(val)
        return arr
    # Try local gz produced earlier in this notebook
    b3_avg = load_b3_mean_from_csv_gz('conjunctive_lazy_rich_b3s.csv.gz')

time_steps, items_n_inferred, b3_size = b3_avg.shape

# Positions to display, used previously
positions = [0.00, 0.10, 0.20, 0.50, 0.65, 0.75, 0.85, 0.90, 1.00]
steps_after_training = time_steps - 1
step_idxs = [max(0, int(p * steps_after_training)) for p in positions]

# Utility to build all concatenated ordered pairs (i!=j)
def build_pairs(X_items):
    vecs, labels, pairs = [], [], []
    for i in range(items_n_inferred):
        for j in range(items_n_inferred):
            if i == j:
                continue
            v = np.concatenate([X_items[i], X_items[j]], axis=0)
            vecs.append(v)
            labels.append(f"({i},{j})")
            pairs.append((i, j))
    V = np.asarray(vecs)
    # mean-center features in each matrix for each step, stabilizes MDS
    V = V - V.mean(axis=0, keepdims=True)
    return V, labels, pairs

# Fit 2D metric MDS on each distance matrix; report Kruskal stress-1 and Kruskal stress (raw)
def fit_mds_with_stress(D, seed):
    mds = manifold.MDS(
        n_components=2,
        dissimilarity='precomputed',
        random_state=seed,
        n_init=4,
        normalized_stress='auto',
    )
    fit_result = mds.fit(D)
    coords = fit_result.embedding_
    stress_raw = getattr(mds, 'stress_', None)
    denom = float(np.sum(D ** 2))
    # Kruskal's stress-1 as in sklearn docs: sqrt( sum((d_ij - d̂_ij)^2) / sum(d_ij^2) )
    stress1 = np.sqrt(stress_raw / denom) if (stress_raw is not None and denom > 0) else np.nan

    # Compute the reconstructed distances
    Dr = euclidean_distances(coords)
    # Use only upper triangle (excluding diagonal)
    orig = D[np.triu_indices_from(D, k=1)]
    rec = Dr[np.triu_indices_from(Dr, k=1)]
    # Kruskal's raw stress
    kruskal_stress = np.sqrt(np.sum((orig - rec) ** 2) / np.sum(orig ** 2)) if np.sum(orig ** 2) > 0 else np.nan

    # For reporting: also include Pearson r between dist matrices
    if len(orig) > 1:
        r, _ = pearsonr(orig, rec)
    else:
        r = np.nan

    return coords, stress1, kruskal_stress, r

seed = mds_seed if 'mds_seed' in globals() else 0

nsteps = len(step_idxs)
ncols = 3
nrows = int(np.ceil(nsteps / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(5.5*ncols, 5*nrows), constrained_layout=True)

for pidx, (pos, step_idx) in enumerate(zip(positions, step_idxs)):
    V, labels, pairs = build_pairs(b3_avg[step_idx])
    D = euclidean_distances(V)
    coords, stress1, kst, r = fit_mds_with_stress(D, seed)
    r_ax = pidx // ncols
    c_ax = pidx % ncols
    ax = axs[r_ax, c_ax] if nrows > 1 else axs[c_ax]
    ax.scatter(coords[:, 0], coords[:, 1], s=18, color='k')
    # full mds
    for i, lab in enumerate(labels):
        ax.text(coords[i, 0], coords[i, 1], lab, fontsize=8, ha='left', va='center')
    ax.axhline(0, color='lightgray', linewidth=1)
    ax.axvline(0, color='lightgray', linewidth=1)
    ax.grid(True, alpha=0.2)
    ax.set_title(
        f"step {step_idx+1} (pos={pos:.2f})\n"
        f"(stress-1={stress1:.3f}, kstress={kst:.3f}, r={r:.3f})"
    )
    ax.set_xlabel('dimension 1')
    ax.set_ylabel('dimension 2')

# Remove any excess unused subplots
for k in range(nsteps, nrows*ncols):
    fig.delaxes(axs.flatten()[k])

plt.show()
